In [ ]:
"""
 https://www.bambooweekly.com/bw-5-ukrainian-exports/
 https://www.bambooweekly.com/bw-5-ukrainian-exports-solution/
 https://github.com/JoergEm/Bamboo-Weekly/tree/main 
"""

In [ ]:
from IPython.display import FileLink, Markdown
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
import seaborn as sns
display(Markdown("Imports ✅"))

In [ ]:
def create_folders(folders: list[str]) -> bool:
    try:
        for folder in folders:
            folderpath: str = os.path.join(os.getcwd(), folder)
            if not os.path.exists(folderpath):
                os.makedirs(folderpath, exist_ok=True)
    except:
        print("Folder {folderpath} could not be created.")
        return False
    else:
        display(Markdown("Folders ✅"))
        return True

In [ ]:
def download_data(url: str, filename: str) -> bool:
    from urllib.request import urlretrieve
    from urllib.error import HTTPError
    try:
        urlretrieve(url, filename)
        return True
    except HTTPError as e:
        if e.code == 403:
            import requests
            try:
                response: requests.Response = requests.get(url)
                with open(filename, 'wb') as f:
                    f.write(response.content)
                    return True
            except:
                print("Could not download Data")
                return False
    return False

In [ ]:
url: str  = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRisnQjodySbp6-XXPGhdsVMp2stg_gyuxw42pP41tuxeic63IARau6bV1TgjLiw_ciAWsTO5LarPqT/pub?output=xlsx"
filename: str  = 'ukrainianexports.xlsx'
folders: list[str]  = ['data', 'results']
filepath: str  = os.path.join(folders[0], filename)
create_folders(folders)

if not os.path.exists(filepath):
    if download_data(url, filepath):
        # data = pd.read_excel(filepath, sheet_name='Data') # 
        df: pd.DataFrame = pd.read_excel(filepath, sheet_name='Data', parse_dates=['Departure'])
        display(Markdown("Data ✅"))
    else:
        display(Markdown('Error ❌'))
else:
    df: pd.DataFrame = pd.read_excel(filepath, sheet_name='Data', parse_dates=['Departure'])
    display(Markdown("Data loaded from existing file ✅"))  

if os.path.exists(filepath):
    display(FileLink(filepath))

In [ ]:
# Create a pivot table, showing how many tons of each commodity (rows) have left each port (columns).
# df.pivot_table(index='Commodity', columns='Departure port', values='Tonnage') # init
df.pivot_table(index='Commodity', columns='Departure port', values='Tonnage', aggfunc='sum')

In [ ]:
# Create a pivot table, showing how many tons of each commodity (columns) were going to each destination country (rows).
# df.pivot_table(columns='Commodity', index='Country', values='Tonnage', aggfunc='sum') # better readability
df.pivot_table(columns='Commodity', index='Country', values='Tonnage', aggfunc='sum').fillna(0)